<a href="https://colab.research.google.com/github/hadri1-bisch/SCM/blob/main/scm_survey_part_3_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Modélisation du questionnaire:**

NB: général
les prix sont indicatif il faudrait réfléchir a des prix plus pértinent

**A) créations des profils**

In [ ]:
import itertools
import pandas as pd
import numpy as np


In [ ]:
# Définition des attributs PAR ALTERNATIVE (design labellisé : Neuf vs Seconde main)

# ---- Alternative "Neuf" : 3 attributs ----
prix_neuf_niveaux = [100, 150, 300]
assortiment_neuf_niveaux = [1, 2]  # 1: Limité, 2: Extensif
garantie_neuf_niveaux = [1, 2]     # 1: Sans retour, 2: Retours 30j

profils_neuf = list(itertools.product(
    prix_neuf_niveaux, assortiment_neuf_niveaux, garantie_neuf_niveaux
))
df_profils_neuf = pd.DataFrame(
    profils_neuf, columns=['Prix', 'Assortiment', 'Garantie']
)

# ---- Alternative "Seconde main" : 4 attributs (Hygiène est spécifique à la SM) ----
prix_sm_niveaux = [100, 150]
hygiene_sm_niveaux = [1, 2]        # 1: Standard (particulier), 2: Certifié professionnel
assortiment_sm_niveaux = [1, 2]    # 1: Limité, 2: Extensif
garantie_sm_niveaux = [1, 2]       # 1: Sans retour, 2: Retours 30j

profils_sm = list(itertools.product(
    prix_sm_niveaux, hygiene_sm_niveaux, assortiment_sm_niveaux, garantie_sm_niveaux
))
df_profils_sm = pd.DataFrame(
    profils_sm, columns=['Prix', 'Hygiene', 'Assortiment', 'Garantie']
)

print(f"Profils Neuf (plein factoriel) : {len(df_profils_neuf)}")
print(df_profils_neuf.to_string(index=False))
print()
print(f"Profils Seconde main (plein factoriel) : {len(df_profils_sm)}")
print(df_profils_sm.to_string(index=False))

Profils Neuf (plein factoriel) : 12
 Prix  Assortiment  Garantie
  100            1         1
  100            1         2
  100            2         1
  100            2         2
  150            1         1
  150            1         2
  150            2         1
  150            2         2
  300            1         1
  300            1         2
  300            2         1
  300            2         2

Profils Seconde main (plein factoriel) : 16
 Prix  Hygiene  Assortiment  Garantie
  100        1            1         1
  100        1            1         2
  100        1            2         1
  100        1            2         2
  100        2            1         1
  100        2            1         2
  100        2            2         1
  100        2            2         2
  150        1            1         1
  150        1            1         2
  150        1            2         1
  150        1            2         2
  150        2            1         1
  150     

B) **Filtrage des profils iréalsites**

Neuf: 3 attributs:
- Prix : 100, 150, 300
- Assortiment : 1 (Limité), 2 (Extensif)
- Garantie : 1 (Sans retour), 2 (Retours 30j)

Seconde main: 4 attributs

- Prix : 100, 150
- Hygiène : 1 (Standard), 2 (Premium/certifié)
- Assortiment : 1 (Limité), 2 (Extensif) — extensif possible uniquement si Hygiène = Premium
- Garantie : 1 (Sans retour), 2 (Retours 30j)

In [ ]:
# Filtrage des profils Seconde main selon la contrainte conditionnelle
def filtre_profils_sm(row):
    # Hygiène standard (particulier) -> pièce unique -> assortiment limité uniquement
    if row['Hygiene'] == 1 and row['Assortiment'] == 2:
        return False
    return True

df_profils_sm_valides = (
    df_profils_sm[df_profils_sm.apply(filtre_profils_sm, axis=1)]
    .reset_index(drop=True)
)

# Aucune contrainte de réalisme identifiée pour le neuf -> tous les profils sont gardés
df_profils_neuf_valides = df_profils_neuf.reset_index(drop=True)

print(f"Profils Neuf valides : {len(df_profils_neuf_valides)}")
print(df_profils_neuf_valides.to_string(index=False))
print()
print(f"Profils Seconde main valides : {len(df_profils_sm_valides)}")
print(df_profils_sm_valides.to_string(index=False))

assert len(df_profils_neuf_valides) == 12, "Le nombre de profils Neuf valides ne correspond pas à l'attendu"
assert len(df_profils_sm_valides) == 12, "Le nombre de profils Seconde main valides ne correspond pas à l'attendu"

Profils Neuf valides : 12
 Prix  Assortiment  Garantie
  100            1         1
  100            1         2
  100            2         1
  100            2         2
  150            1         1
  150            1         2
  150            2         1
  150            2         2
  300            1         1
  300            1         2
  300            2         1
  300            2         2

Profils Seconde main valides : 12
 Prix  Hygiene  Assortiment  Garantie
  100        1            1         1
  100        1            1         2
  100        2            1         1
  100        2            1         2
  100        2            2         1
  100        2            2         2
  150        1            1         1
  150        1            1         2
  150        2            1         1
  150        2            1         2
  150        2            2         1
  150        2            2         2


**C) Question a insérer dans le questionnaires**

créations de toutes les paires candidates:

In [ ]:
from itertools import product

In [ ]:
# Générer toutes les paires : seconde main vs neuf
paires = []

for (idx_sm, profil_sm), (idx_neuf, profil_neuf) in product(
    df_profils_sm_valides.iterrows(),
    df_profils_neuf_valides.iterrows()
):
    paires.append({
        "Prix_SM": profil_sm["Prix"],
        "Hygiene_SM": profil_sm["Hygiene"],
        "Tailles_SM": profil_sm["Assortiment"],
        "Retours_SM": profil_sm["Garantie"],
        "Prix_neuf": profil_neuf["Prix"],
        "Tailles_neuf": profil_neuf["Assortiment"],
        "Retours_neuf": profil_neuf["Garantie"]
    })

df_paires_candidates = pd.DataFrame(paires)

print(f"Nombre de paires candidates : {len(df_paires_candidates)}")
print(df_paires_candidates.to_string(index=False))

assert len(df_paires_candidates) == 144, "Le nombre de paires candidates ne correspond pas à l'attendu"

Nombre de paires candidates : 144
 Prix_SM  Hygiene_SM  Tailles_SM  Retours_SM  Prix_neuf  Tailles_neuf  Retours_neuf
     100           1           1           1        100             1             1
     100           1           1           1        100             1             2
     100           1           1           1        100             2             1
     100           1           1           1        100             2             2
     100           1           1           1        150             1             1
     100           1           1           1        150             1             2
     100           1           1           1        150             2             1
     100           1           1           1        150             2             2
     100           1           1           1        300             1             1
     100           1           1           1        300             1             2
     100           1           1          

In [ ]:
# Supprimer les questions évidentes :
# une option est au moins aussi bonne sur le prix, les tailles ET les retours
def est_evident(row):
    sm_mieux = (row["Prix_SM"] <= row["Prix_neuf"]
                and row["Tailles_SM"] >= row["Tailles_neuf"]
                and row["Retours_SM"] >= row["Retours_neuf"])
    neuf_mieux = (row["Prix_neuf"] <= row["Prix_SM"]
                  and row["Tailles_neuf"] >= row["Tailles_SM"]
                  and row["Retours_neuf"] >= row["Retours_SM"])
    return sm_mieux or neuf_mieux

df_paires_candidates = (
    df_paires_candidates[~df_paires_candidates.apply(est_evident, axis=1)]
    .reset_index(drop=True)
)
print("Paires restantes :", len(df_paires_candidates))

Paires restantes : 51


**Choix des questions**
but min d-error

In [ ]:
pip install choicedesign

In [ ]:
import choicedesign as cd

méthode d-efficency sans prior

In [ ]:
from choicedesign.design import EffDesign
from choicedesign.expressions import Attribute, Parameter, ASC

# --------------------------------------------------
# 1. Définir les attributs des profils existants
# --------------------------------------------------

prix_sm = Attribute("Prix_SM", [100, 150])
hygiene_sm = Attribute("Hygiene_SM", [1, 2])
tailles_sm = Attribute("Tailles_SM", [1, 2])
retours_sm = Attribute("Retours_SM", [1, 2])

prix_neuf = Attribute("Prix_neuf", [100, 150, 300])
tailles_neuf = Attribute("Tailles_neuf", [1, 2])
retours_neuf = Attribute("Retours_neuf", [1, 2])

# --------------------------------------------------
# 2. Définir les paramètres du modèle
# Priors nuls : tous les coefficients commencent à 0
# --------------------------------------------------

beta_prix_150 = Parameter("beta_prix_150", 0.0)
beta_prix_300 = Parameter("beta_prix_300", 0.0)
beta_tailles = Parameter("beta_tailles", 0.0)
beta_retours = Parameter("beta_retours", 0.0)
beta_hygiene = Parameter("beta_hygiene", 0.0)

# Constante spécifique à l'alternative seconde main
asc_sm = ASC("asc_sm", 0.0)

# --------------------------------------------------
# 3. Définir les fonctions d'utilité
# Le prix est catégoriel, avec 100 CHF comme référence
# L'hygiène n'existe que côté seconde main (pas d'équivalent pour le neuf)
# --------------------------------------------------

V_sm = (
    asc_sm
    + beta_prix_150 * (prix_sm == 150)
    + beta_hygiene * (hygiene_sm == 2)
    + beta_tailles * (tailles_sm == 2)
    + beta_retours * (retours_sm == 2)
)

V_neuf = (
    beta_prix_150 * (prix_neuf == 150)
    + beta_prix_300 * (prix_neuf == 300)
    + beta_tailles * (tailles_neuf == 2)
    + beta_retours * (retours_neuf == 2)
)

V = {
    1: V_sm,
    2: V_neuf
}

NB: cet algo n'est pas optimal il teste pas tout car sinon plus de 12mn a run.... **vu que c'est tiré en random est que y'a que 70/144 echantillons je me demande pas si l'on peut obtenir des pool de question différentes, à clarifier **
perso meilleure d-error etait de 0.46

c'est en changeant ces paramètres que l'on peut opti:

nombre_depart = 3        
iterations_max = 15      
taille_echantillon = 60  

In [ ]:
# --------------------------------------------------
# 4. Préparer les 144 paires candidates
# --------------------------------------------------

colonnes_modele = [
    "Prix_SM",
    "Hygiene_SM",
    "Tailles_SM",
    "Retours_SM",
    "Prix_neuf",
    "Tailles_neuf",
    "Retours_neuf"
]

candidates = df_paires_candidates.reset_index(drop=True).copy()

# --------------------------------------------------
# 5. Créer l'évaluateur D-efficient
# --------------------------------------------------

design = EffDesign(
    X=[
        prix_sm,
        hygiene_sm,
        tailles_sm,
        retours_sm,
        prix_neuf,
        tailles_neuf,
        retours_neuf
    ],
    ncs=12
)


def evaluer_selection(indices):
    plan = candidates.iloc[indices][colonnes_modele].copy()

    # au moins 4 questions avec hygiène standard
    if (plan["Hygiene_SM"] == 1).sum() < 4:
        return np.inf, 0.0

    plan.insert(0, "CS", range(1, len(plan) + 1))

    d_error, equilibre = design.evaluate(
        plan,
        V=V,
        criterion="d"
    )

    return float(d_error), float(equilibre)

# --------------------------------------------------
# 6. Recherche locale avec plusieurs départs aléatoires
#    (version limitée en temps : moins de départs/itérations,
#    et échantillonnage des candidats testés à chaque swap)
# --------------------------------------------------

rng = np.random.default_rng(42)

meilleur_plan = None
meilleur_d_error = np.inf
meilleur_equilibre = None

nombre_depart = 5       # réduit de 5 à 3
iterations_max = 25      # réduit de 30 à 15
taille_echantillon = 70  # au lieu de tester tous les candidats restants, on en teste un échantillon

for depart in range(nombre_depart):

    selection = list(
        rng.choice(len(candidates), size=12, replace=False)
    )

    d_error, equilibre = evaluer_selection(selection)

    for iteration in range(iterations_max):

        indices_selectionnes = set(selection)
        candidats_restants = [
            i for i in range(len(candidates)) if i not in indices_selectionnes
        ]
        candidats_testes = rng.choice(
            candidats_restants,
            size=min(taille_echantillon, len(candidats_restants)),
            replace=False
        )

        meilleur_echange = None
        meilleur_d_local = d_error
        meilleur_eq_local = equilibre

        # Tester le remplacement d'une question par une autre (échantillon seulement)
        for position in range(12):

            for candidate_idx in candidats_testes:

                essai = selection.copy()
                essai[position] = candidate_idx

                d_essai, eq_essai = evaluer_selection(essai)

                if (
                    np.isfinite(d_essai)
                    and d_essai < meilleur_d_local - 1e-10
                ):
                    meilleur_d_local = d_essai
                    meilleur_eq_local = eq_essai
                    meilleur_echange = essai

        # Arrêter si aucune substitution n'améliore le plan
        if meilleur_echange is None:
            break

        selection = meilleur_echange
        d_error = meilleur_d_local
        equilibre = meilleur_eq_local

    print(f"Départ {depart + 1}/{nombre_depart} terminé — D-error : {d_error:.4f}")

    if d_error < meilleur_d_error:
        meilleur_d_error = d_error
        meilleur_equilibre = equilibre
        meilleur_plan = selection.copy()

# --------------------------------------------------
# 7. Afficher les 12 questions sélectionnées
# --------------------------------------------------

df_questions = candidates.iloc[meilleur_plan].copy()
df_questions = df_questions.reset_index(drop=True)

df_questions.insert(
    0, "Question", range(1, len(df_questions) + 1)
)

print("Nombre de questions :", len(df_questions))
print("D-error final :", meilleur_d_error)
print("Équilibre des utilités (%) :", meilleur_equilibre)

display(df_questions)

Départ 1/5 terminé — D-error : 0.5193
Départ 2/5 terminé — D-error : 0.5255
Départ 3/5 terminé — D-error : inf
Départ 4/5 terminé — D-error : 0.5173
Départ 5/5 terminé — D-error : 0.5173
Nombre de questions : 12
D-error final : 0.517341749354424
Équilibre des utilités (%) : 100.0


,Question,Prix_SM,Hygiene_SM,Tailles_SM,Retours_SM,Prix_neuf,Tailles_neuf,Retours_neuf
0,1,150,2,1,2,100,2,1
1,2,150,1,1,2,300,2,1
2,3,150,2,2,2,100,1,1
3,4,100,1,1,2,300,2,1
4,5,100,1,1,1,150,2,2
5,6,100,2,2,1,300,1,2
6,7,100,2,1,2,150,2,1
7,8,150,2,2,1,100,1,2
8,9,100,2,1,1,150,2,2
9,10,150,2,1,1,300,2,2


Question avec prior, j'ai mis l'accent sur hygiène et prix

In [ ]:
from choicedesign.design import EffDesign
from choicedesign.expressions import Attribute, Parameter, ASC

# --------------------------------------------------
# 1. Attributs (inchangé)
# --------------------------------------------------
prix_sm = Attribute("Prix_SM", [100, 150])
hygiene_sm = Attribute("Hygiene_SM", [1, 2])
tailles_sm = Attribute("Tailles_SM", [1, 2])
retours_sm = Attribute("Retours_SM", [1, 2])

prix_neuf = Attribute("Prix_neuf", [100, 150, 300])
tailles_neuf = Attribute("Tailles_neuf", [1, 2])
retours_neuf = Attribute("Retours_neuf", [1, 2])

# --------------------------------------------------
# 2. Paramètres
# Prix LINÉAIRE : un seul coefficient "par franc"
# (remplace beta_prix_150 et beta_prix_300)
# Priors supposés (signe attendu), pas tirés de la littérature
# --------------------------------------------------
beta_prix = Parameter("beta_prix", -0.01)       # effet de chaque franc
beta_tailles = Parameter("beta_tailles", 0.3)   # ≈ 30 CHF
beta_retours = Parameter("beta_retours", 0.3)   # ≈ 30 CHF
beta_hygiene = Parameter("beta_hygiene", 0.5)   # ≈ 50 CHF

asc_sm = ASC("asc_sm", 0.0)

# --------------------------------------------------
# 3. Fonctions d'utilité
# Le prix entre directement comme un nombre
# --------------------------------------------------
V_sm = (
    asc_sm
    + beta_prix * prix_sm
    + beta_hygiene * (hygiene_sm == 2)
    + beta_tailles * (tailles_sm == 2)
    + beta_retours * (retours_sm == 2)
)

V_neuf = (
    beta_prix * prix_neuf
    + beta_tailles * (tailles_neuf == 2)
    + beta_retours * (retours_neuf == 2)
)

V = {
    1: V_sm,
    2: V_neuf
}

candidats:

In [ ]:
# --------------------------------------------------
# 4. Préparer les 144 paires candidates
# --------------------------------------------------

colonnes_modele = [
    "Prix_SM",
    "Hygiene_SM",
    "Tailles_SM",
    "Retours_SM",
    "Prix_neuf",
    "Tailles_neuf",
    "Retours_neuf"
]

candidates = df_paires_candidates.reset_index(drop=True).copy()


# --------------------------------------------------
# 5. Créer l'évaluateur D-efficient
# --------------------------------------------------

design = EffDesign(
    X=[
        prix_sm,
        hygiene_sm,
        tailles_sm,
        retours_sm,
        prix_neuf,
        tailles_neuf,
        retours_neuf
    ],
    ncs=12
)


def evaluer_selection(indices):
    plan = candidates.iloc[indices][colonnes_modele].copy()

    # au moins 4 questions avec hygiène standard
    if (plan["Hygiene_SM"] == 1).sum() < 4:
        return np.inf, 0.0

    plan.insert(0, "CS", range(1, len(plan) + 1))

    d_error, equilibre = design.evaluate(
        plan,
        V=V,
        criterion="d"
    )

    return float(d_error), float(equilibre)
# --------------------------------------------------
# 6. Recherche locale avec plusieurs départs aléatoires
#    (version limitée en temps : échantillonnage des candidats testés à chaque swap)
# --------------------------------------------------

rng = np.random.default_rng(42)

meilleur_plan = None
meilleur_d_error = np.inf
meilleur_equilibre = None

nombre_depart = 5
iterations_max = 25
taille_echantillon = 70

for depart in range(nombre_depart):

    # tirer un départ qui respecte déjà la règle des 4 "standard"
    while True:
        selection = list(
            rng.choice(len(candidates), size=12, replace=False)
        )
        if (candidates.iloc[selection]["Hygiene_SM"] == 1).sum() >= 4:
            break

    d_error, equilibre = evaluer_selection(selection)

    for iteration in range(iterations_max):

        indices_selectionnes = set(selection)
        candidats_restants = [
            i for i in range(len(candidates)) if i not in indices_selectionnes
        ]
        candidats_testes = rng.choice(
            candidats_restants,
            size=min(taille_echantillon, len(candidats_restants)),
            replace=False
        )

        meilleur_echange = None
        meilleur_d_local = d_error
        meilleur_eq_local = equilibre

        for position in range(12):

            for candidate_idx in candidats_testes:

                essai = selection.copy()
                essai[position] = candidate_idx

                d_essai, eq_essai = evaluer_selection(essai)

                if (
                    np.isfinite(d_essai)
                    and d_essai < meilleur_d_local - 1e-10
                ):
                    meilleur_d_local = d_essai
                    meilleur_eq_local = eq_essai
                    meilleur_echange = essai

        if meilleur_echange is None:
            break

        selection = meilleur_echange
        d_error = meilleur_d_local
        equilibre = meilleur_eq_local

    print(f"Départ {depart + 1}/{nombre_depart} terminé — D-error : {d_error:.4f}")

    if d_error < meilleur_d_error:
        meilleur_d_error = d_error
        meilleur_equilibre = equilibre
        meilleur_plan = selection.copy()

# --------------------------------------------------
# 7. Afficher les 12 questions sélectionnées
# --------------------------------------------------

df_questions = candidates.iloc[meilleur_plan].copy()
df_questions = df_questions.reset_index(drop=True)

df_questions.insert(
    0, "Question", range(1, len(df_questions) + 1)
)

print("Nombre de questions :", len(df_questions))
print("D-error final :", meilleur_d_error)
print("Équilibre des utilités (%) :", meilleur_equilibre)

display(df_questions)

Départ 1/5 terminé — D-error : 0.0524
Départ 2/5 terminé — D-error : 0.0524
Départ 3/5 terminé — D-error : 0.0524
Départ 4/5 terminé — D-error : 0.0524
Départ 5/5 terminé — D-error : 0.0524
Nombre de questions : 12
D-error final : 0.0524140246196172
Équilibre des utilités (%) : 80.09529681362365


,Question,Prix_SM,Hygiene_SM,Tailles_SM,Retours_SM,Prix_neuf,Tailles_neuf,Retours_neuf
0,1,150,2,1,2,100,2,1
1,2,100,1,1,2,300,2,1
2,3,150,1,1,2,300,2,1
3,4,100,2,2,1,150,1,2
4,5,100,2,1,2,100,2,1
5,6,100,1,1,1,300,2,2
6,7,150,2,2,1,300,1,2
7,8,150,2,1,2,150,2,1
8,9,150,2,2,1,100,1,2
9,10,150,2,2,2,100,1,1


In [ ]:
# Dictionnaires de correspondance code -> libellé
labels_hygiene = {1: "Standard", 2: "Premium/certifié"}
labels_assortiment = {1: "Limité", 2: "Extensif"}
labels_garantie = {1: "Sans retour", 2: "Retours 30j"}

df_questions_lisible = df_questions.copy()

df_questions_lisible["Hygiene_SM"] = df_questions_lisible["Hygiene_SM"].replace(labels_hygiene)
df_questions_lisible["Tailles_SM"] = df_questions_lisible["Tailles_SM"].replace(labels_assortiment)
df_questions_lisible["Retours_SM"] = df_questions_lisible["Retours_SM"].replace(labels_garantie)
df_questions_lisible["Tailles_neuf"] = df_questions_lisible["Tailles_neuf"].replace(labels_assortiment)
df_questions_lisible["Retours_neuf"] = df_questions_lisible["Retours_neuf"].replace(labels_garantie)

display(df_questions_lisible)

,Question,Prix_SM,Hygiene_SM,Tailles_SM,Retours_SM,Prix_neuf,Tailles_neuf,Retours_neuf
0,1,150,Premium/certifié,Limité,Retours 30j,100,Extensif,Sans retour
1,2,100,Standard,Limité,Retours 30j,300,Extensif,Sans retour
2,3,150,Standard,Limité,Retours 30j,300,Extensif,Sans retour
3,4,100,Premium/certifié,Extensif,Sans retour,150,Limité,Retours 30j
4,5,100,Premium/certifié,Limité,Retours 30j,100,Extensif,Sans retour
5,6,100,Standard,Limité,Sans retour,300,Extensif,Retours 30j
6,7,150,Premium/certifié,Extensif,Sans retour,300,Limité,Retours 30j
7,8,150,Premium/certifié,Limité,Retours 30j,150,Extensif,Sans retour
8,9,150,Premium/certifié,Extensif,Sans retour,100,Limité,Retours 30j
9,10,150,Premium/certifié,Extensif,Retours 30j,100,Limité,Sans retour


interpretation:
Deux chemins possibles
Garder tel quel (ce que je recommanderais par défaut) : c’est la pratique standard, bien justifiée statistiquement, et l’impact sur la validité perçue est généralement mineur pour ce type d’enquête.
Rendre beta_tailles et beta_retours spécifiques à chaque alternative (comme beta_hygiene l’est déjà) — c’est-à-dire avoir beta_tailles_sm et beta_tailles_neuf séparés, idem pour retours. Ça lèverait la contrainte de “contraste obligatoire” et permettrait des paires avec des niveaux identiques entre SM et neuf, donc un design visuellement plus “naturel”. Le coût : 2 paramètres supplémentaires à estimer (6 au lieu de 4 au total pour ces deux attributs), ce qui demande plus de répondants pour la même précision, et complique l’interprétation (tu ne pourrais plus dire “l’assortiment compte autant pour le neuf que pour la seconde main” — ce serait justement ce que tu testerais).

Note audrey: Gemini dit que c'est mieux de séparer.Puisque vous êtes sur un design labelled (Neuf vs Seconde main), la valeur psychologique d'une politique de retour n'est pas la même. Un consommateur valorisera probablement beaucoup plus la possibilité de retour pour un vêtement d'occasion (pour contrer le risque de performance ou sanitaire) que pour un vêtement neuf de fast fashion. Séparer ces betas vous permettra de prouver statistiquement que la garantie est un levier plus puissant pour la seconde main.